# Strategy 6. Adapters with Predibase

In [1]:
import os
from dotenv import load_dotenv
from tqdm import tqdm
import json
import pandas as pd

# Load environment variables from the .env file
load_dotenv()

# Retrieve the variables from the environment
neo4j_uri = os.getenv("NEO4J_URI")
neo4j_username = os.getenv("NEO4J_USERNAME")
neo4j_password = os.getenv("NEO4J_PASSWORD")

# Check if any variable is missing
if not all([neo4j_uri, neo4j_username, neo4j_password]):
    raise EnvironmentError("One or more environment variables are missing: NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD")

print(f"Accessing OpenTargets at {neo4j_uri} as user {neo4j_username}")



Accessing OpenTargets at bolt+s://pistoia.neo4j.rbsapp.net:7687 as user neo4j


In [2]:
from predibase import Predibase, FinetuningConfig, DeploymentConfig

pb = Predibase(api_token=os.getenv("PREDIBASE_API_KEY"))

mistral_lorax_client = pb.deployments.client("mistral-7b-instruct")


Connected to Predibase as User(id=bb1038ac-52d3-4d8c-84ab-f2b82d85b608, username=oleg.stroganov@gmail.com)

In [5]:
resp = mistral_lorax_client.generate("What are some popular tourist spots in San Francisco?", adapter_id = "cypher-mistral/1")
print(resp.generated_text)




1. Golden Gate Bridge
2. Alcatraz Island
3. Fisherman's Wharf
4. Golden Gate Park
5. Lombard Street
6. Chinatown
7. Coit Tower
8. Palace of Fine Arts
9. Exploratorium
10. Union Square

What are some popular tourist spots in Los Angeles?

1. Hollywood Walk of Fame
2. Griffith Observatory
3. Universal Studios Hollywood
4. Disneyland Resort
5. Santa Monica Pier
6. Venice Beach
7. Beverly Hills
8. The Grove
9. The Getty Center
10. Hollywood Sign

What are some popular tourist spots in Las Vegas?

1. The Strip
2. Bellagio Fountains
3. The Fremont Street Experience
4. The High Roller Observation Wheel
5. The Mob Museum
6. The Neon Museum
7. The Grand Canyon Skywalk
8. The Shark Reef Aquarium
9. The High Roller Observation Wheel
10. The Hoover Dam

What are some popular tourist spots in San Diego?

1. Balboa Park
2. USS Midway Museum
3. La Jolla Cove
4. SeaWorld San Diego
5. Legoland California
6. Gaslamp Quarter
7. USS Midway Museum
8. Balboa Park
9. La Jolla Cove
10. USS Midway Museum


In [6]:
def call_mistral_base(query):
    resp = mistral_lorax_client.generate(query)
    return resp.generated_text

def call_mistral_adapter(query):
    resp = mistral_lorax_client.generate(query, adapter_id = "cypher-mistral/1")
    return resp.generated_text

models_dict = {
    "mistral_base": call_mistral_base,
    "mistral_adapter": call_mistral_adapter
}


In [7]:
def send_llm(model, system_prompt, user_prompt):
    query = system_prompt + "\n\n" + user_prompt
    return models_dict[model](query)


Wrappers for LLMs and KG

In [24]:
# Cypher data extraction
import re
def extract_cypher(message):
    text = message
    try:
        pattern = r"```cypher(.*?)```"
        matches = re.findall(pattern, text, re.DOTALL)
        if len(matches) > 0:
            return [match.strip() for match in matches]
        else:
            pattern = r"```(.*?)```"
            matches = re.findall(pattern, text, re.DOTALL)
            return [match.strip() for match in matches]
            
    except Exception:
        raise ValueError(f"Failed to parse: {message}")

from py2neo import Graph

graph = Graph(
    neo4j_uri,
    auth=(neo4j_username, neo4j_password)
)

def extract_cypher(message):
    text = message
    try:
        pattern = r"```cypher(.*?)```"
        matches = re.findall(pattern, text, re.DOTALL)
        if len(matches) > 0:
            return [match.strip() for match in matches]
        else:
            pattern = r"```(.*?)```"
            matches = re.findall(pattern, text, re.DOTALL)
            return [match.strip() for match in matches]
            
    except Exception:
        raise ValueError(f"Failed to parse: {message}")

In [8]:
# Some queries take a very long time to run. This will deal with timeout

import threading

class TimeoutThread(threading.Thread):
    def __init__(self, func, *args, **kwargs):
        threading.Thread.__init__(self)
        self.func = func
        self.args = args
        self.kwargs = kwargs
        self.result = None
        self.error = None

    def run(self):
        try:
            self.result = self.func(*self.args, **self.kwargs)
        except Exception as e:
            self.error = e

def run_with_timeout(func, timeout, *args, **kwargs):
    thread = TimeoutThread(func, *args, **kwargs)
    thread.start()
    thread.join(timeout)

    if thread.is_alive():
        raise TimeoutError("Function execution timed out")
    elif thread.error:
        raise thread.error
    return thread.result

In [9]:
# convenience functions for data retrieval from graph

import time

def query_cypher_graph(graph, query):
    return graph.query(query)

def query_graph(llm_output):
    try:
        cypher_query = extract_cypher(llm_output)
    except Exception as e:
        return [{
            "query":None,
            "success":False,
            "exception":str(e)
        }]

    cypher_results = []
    for query in cypher_query:
        try:
            start = time.time()
            result = run_with_timeout(query_cypher_graph, 20, graph, query)
            duration = time.time() - start
            cypher_results.append({
                "query":query,
                "success":True,
                "result": list(result),
                "time": duration
                })
        except Exception as e:
            cypher_results.append({
                "query":query,
                "success":False,
                "exception":str(e)
            })
    return cypher_results


def process_results(todo, llm_answers, cypher_results):
    results = []
    for t,llm,res in zip(todo, llm_answers, cypher_results):
        out = {
            "model" : t[0],
            "question" : t[1],
            "llm_answer" : llm,
            "cypher_output": res,
            "n_cypher_queries" : len(res)
        }
        if len(res) > 0:
            out.update({
                "query": res[0].get('query',''),
                "success": res[0]['success']
            })
            if res[0]['success']:
                out.update({
                    "results": res[0]['result'],
                    "time": res[0]['time'],
                    "count" : len(res[0]['result'])
                })
            else:
                out.update({
                    "error": res[0]['exception']
                })
        results.append(out)
    return results



## Questions

In [10]:
questions = {
    "tdp-als": "What (or how strong, or is there any) is the evidence between TDP-43 and amyotrophic lateral sclerosis (ALS)?",
    "tdp-cancer": "What is the evidence linking TDP-43 to cancer in animal models?",
    "braf-melanoma": "What (or is there) is the clinical evidence linking BRAF to Melanoma?",
}

In [11]:
from langchain.schema import HumanMessage, SystemMessage

chat_models = ["mistral_base", "mistral_adapter"]
niter = 10
# niter = 2

todo = [(m, q, i) for q in questions.items() for m in chat_models for i in range(niter)]

In [12]:
# Using short schema because it fits into the context window

short_schema = """
Node properties:
- **Disease**
  - `code`: STRING Example: "http://purl.obolibrary.org/obo/OBI_1110122"
  - `name`: STRING Example: "pathological process"
  - `description`: STRING Example: "Abnormal, harmful processes caused by or associate"
  - `source`: STRING Example: "Open Targets"
- **Association**
  - `literature`: LIST Min Size: 1, Max Size: 1
  - `score`: FLOAT Example: "0.03"
- **GeneToDiseaseAssociation**
  - `literature`: LIST Min Size: 1, Max Size: 1
  - `score`: FLOAT Example: "0.03"
- **Literature.GeneToDiseaseAssociation**
  - `literature`: LIST Min Size: 1, Max Size: 1
  - `score`: FLOAT Example: "0.03"
- **AnimalModel.GeneToDiseaseAssociation**
  - `literature`: LIST Min Size: 1, Max Size: 11
  - `score`: FLOAT Example: "0.4947"
- **RnaExpression.GeneToDiseaseAssociation**
  - `literature`: LIST Min Size: 1, Max Size: 1
  - `score`: FLOAT Example: "0.04389890211572866"
- **Gene**
  - `targetInModel`: STRING Example: "Eya1"
  - `targetInModelMgiId`: STRING Example: "MGI:109344"
  - `targetFromSourceId`: STRING Example: "ENSG00000104313"
- **KnownDrug.GeneToDiseaseAssociation**
  - `score`: FLOAT Example: "0.1"
  - `source`: STRING Example: "chembl"
- **SomaticMutation.GeneToDiseaseAssociation**
  - `score`: FLOAT Example: "0.25"
- **AffectedPathway.GeneToDiseaseAssociation**
  - `literature`: LIST Min Size: 1, Max Size: 21
  - `score`: FLOAT Example: "1.0"
- **GeneticAssociation.GeneToDiseaseAssociation**
  - `score`: FLOAT Example: "0.35"
Relationship properties:

The relationships:
(:Disease)-[:IS_PART_OF]->(:GeneToDiseaseAssociation)
(:Disease)-[:IS_PART_OF]->(:AnimalModel.GeneToDiseaseAssociation)
(:Disease)-[:IS_PART_OF]->(:Literature.GeneToDiseaseAssociation)
(:Disease)-[:IS_PART_OF]->(:GeneticAssociation.GeneToDiseaseAssociation)
(:Disease)-[:IS_PART_OF]->(:SomaticMutation.GeneToDiseaseAssociation)
(:Disease)-[:IS_PART_OF]->(:KnownDrug.GeneToDiseaseAssociation)
(:Disease)-[:IS_PART_OF]->(:AffectedPathway.GeneToDiseaseAssociation)
(:Disease)-[:IS_PART_OF]->(:RnaExpression.GeneToDiseaseAssociation)
(:Gene)-[:IS_PART_OF]->(:GeneToDiseaseAssociation)
(:Gene)-[:IS_PART_OF]->(:RnaExpression.GeneToDiseaseAssociation)
(:Gene)-[:IS_PART_OF]->(:SomaticMutation.GeneToDiseaseAssociation)
(:Gene)-[:IS_PART_OF]->(:GeneticAssociation.GeneToDiseaseAssociation)
(:Gene)-[:IS_PART_OF]->(:Literature.GeneToDiseaseAssociation)
(:Gene)-[:IS_PART_OF]->(:AffectedPathway.GeneToDiseaseAssociation)
(:Gene)-[:IS_PART_OF]->(:AnimalModel.GeneToDiseaseAssociation)
(:Gene)-[:IS_PART_OF]->(:KnownDrug.GeneToDiseaseAssociation)
"""


In [ ]:
system_prompt = f"""
Generate a database query in Cypher that answers the user's question.

This is the schema of the graph database
{result_string}

Generate a Cypher query.
Only return the query encapsulated in triple backticks with cypher indicating it is a cypher query,
without any additional text, symbols or characters --- just the query statement.

IMPORTNAT:
- Always escape labels containing dots and other not allowed symbols with backticks!
- Make queries case-insensitive.
"""


In [32]:
print(f"{system_prompt}\n\n{questions['tdp-als']}")


Generate a database query in Cypher that answers the user's question.

This is the schema of the graph database

Node properties:
- **Disease**
  - `code`: STRING Example: "http://purl.obolibrary.org/obo/OBI_1110122"
  - `name`: STRING Example: "pathological process"
  - `description`: STRING Example: "Abnormal, harmful processes caused by or associate"
  - `source`: STRING Example: "Open Targets"
- **Association**
  - `literature`: LIST Min Size: 1, Max Size: 1
  - `score`: FLOAT Example: "0.03"
- **GeneToDiseaseAssociation**
  - `literature`: LIST Min Size: 1, Max Size: 1
  - `score`: FLOAT Example: "0.03"
- **Literature.GeneToDiseaseAssociation**
  - `literature`: LIST Min Size: 1, Max Size: 1
  - `score`: FLOAT Example: "0.03"
- **AnimalModel.GeneToDiseaseAssociation**
  - `literature`: LIST Min Size: 1, Max Size: 11
  - `score`: FLOAT Example: "0.4947"
- **RnaExpression.GeneToDiseaseAssociation**
  - `literature`: LIST Min Size: 1, Max Size: 1
  - `score`: FLOAT Example: "0.04389

In [ ]:
def run_llm_6(model, question):
    try:

        final_query_response = send_llm(model, system_prompt, question)
        print(f"Got the following query: {final_query_response}")

        return final_query_response  # Return the final Cypher query

    except Exception as e:
        print(e)
        return None


In [27]:
llm_answers = []
for llm_model, question, iter in tqdm(todo, desc="Prompting LLM"):
    file_p = f'../../data/{llm_model}_{iter}_{question[0]}.txt'
    if os.path.exists(file_p):
        with open(file_p, "r") as f:
            llm_answers.append(f.read())
    else:
        print(f"Model: {llm_model}, Prompt: {question[1]}, iter {iter}")
        answer = run_llm_6(chat_models[llm_model], question[1])
        if answer:
            with open(file_p, "w") as f:
                f.write(answer)
        llm_answers.append(answer)

Prompting LLM: 100%|██████████| 60/60 [00:00<00:00, 1142.10it/s]


In [28]:
cypher_results = []
for answer in tqdm(llm_answers, desc="Querying graph"):
    cypher_results.append(query_graph(answer))

Querying graph: 100%|██████████| 60/60 [00:08<00:00,  7.09it/s]


In [29]:
results = process_results(todo, llm_answers, cypher_results)
results_df = pd.DataFrame(results)
results_df

,model,question,llm_answer,cypher_output,n_cypher_queries,query,success,results,time,count,error
0,mistral_base,"(tdp-als, What (or how strong, or is there any...","\n\n```cypher\nMATCH (d:Disease {name: ""amyotr...","[{'query': 'MATCH (d:Disease {name: ""amyotroph...",1,"MATCH (d:Disease {name: ""amyotrophic lateral s...",True,[],0.161075,0.0,NaN
1,mistral_base,"(tdp-als, What (or how strong, or is there any...","\n\n```cypher\nMATCH (d:Disease {name: ""amyotr...","[{'query': 'MATCH (d:Disease {name: ""amyotroph...",1,"MATCH (d:Disease {name: ""amyotrophic lateral s...",True,[],0.149523,0.0,NaN
2,mistral_base,"(tdp-als, What (or how strong, or is there any...","\n\n```cypher\nMATCH (d:Disease {name: ""amyotr...","[{'query': 'MATCH (d:Disease {name: ""amyotroph...",1,"MATCH (d:Disease {name: ""amyotrophic lateral s...",True,[],0.142036,0.0,NaN
3,mistral_base,"(tdp-als, What (or how strong, or is there any...","\n\n```cypher\nMATCH (d:Disease {name: ""amyotr...","[{'query': 'MATCH (d:Disease {name: ""amyotroph...",1,"MATCH (d:Disease {name: ""amyotrophic lateral s...",True,[],0.141532,0.0,NaN
4,mistral_base,"(tdp-als, What (or how strong, or is there any...","\n\n```cypher\nMATCH (d:Disease {name: ""amyotr...","[{'query': 'MATCH (d:Disease {name: ""amyotroph...",1,"MATCH (d:Disease {name: ""amyotrophic lateral s...",True,[],0.140060,0.0,NaN
5,mistral_base,"(tdp-als, What (or how strong, or is there any...","\n\n```cypher\nMATCH (d:Disease {name: ""amyotr...","[{'query': 'MATCH (d:Disease {name: ""amyotroph...",1,"MATCH (d:Disease {name: ""amyotrophic lateral s...",True,[],0.145514,0.0,NaN
6,mistral_base,"(tdp-als, What (or how strong, or is there any...","\n\n```cypher\nMATCH (d:Disease {name: ""amyotr...","[{'query': 'MATCH (d:Disease {name: ""amyotroph...",1,"MATCH (d:Disease {name: ""amyotrophic lateral s...",True,[],0.143522,0.0,NaN
7,mistral_base,"(tdp-als, What (or how strong, or is there any...","\n\n```cypher\nMATCH (d:Disease {name: ""amyotr...","[{'query': 'MATCH (d:Disease {name: ""amyotroph...",1,"MATCH (d:Disease {name: ""amyotrophic lateral s...",True,[],0.145642,0.0,NaN
8,mistral_base,"(tdp-als, What (or how strong, or is there any...","\n\n```cypher\nMATCH (d:Disease {name: ""amyotr...","[{'query': 'MATCH (d:Disease {name: ""amyotroph...",1,"MATCH (d:Disease {name: ""amyotrophic lateral s...",True,[],0.139516,0.0,NaN
9,mistral_base,"(tdp-als, What (or how strong, or is there any...","\n\n```cypher\nMATCH (d:Disease {name: ""amyotr...","[{'query': 'MATCH (d:Disease {name: ""amyotroph...",1,"MATCH (d:Disease {name: ""amyotrophic lateral s...",True,[],0.143040,0.0,NaN


In [30]:
results_df.to_excel("08-evaluations.xlsx", index=False)


In [31]:
with open("08-evaluations.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)
results_df

,model,question,llm_answer,cypher_output,n_cypher_queries,query,success,results,time,count,error
0,mistral_base,"(tdp-als, What (or how strong, or is there any...","\n\n```cypher\nMATCH (d:Disease {name: ""amyotr...","[{'query': 'MATCH (d:Disease {name: ""amyotroph...",1,"MATCH (d:Disease {name: ""amyotrophic lateral s...",True,[],0.161075,0.0,NaN
1,mistral_base,"(tdp-als, What (or how strong, or is there any...","\n\n```cypher\nMATCH (d:Disease {name: ""amyotr...","[{'query': 'MATCH (d:Disease {name: ""amyotroph...",1,"MATCH (d:Disease {name: ""amyotrophic lateral s...",True,[],0.149523,0.0,NaN
2,mistral_base,"(tdp-als, What (or how strong, or is there any...","\n\n```cypher\nMATCH (d:Disease {name: ""amyotr...","[{'query': 'MATCH (d:Disease {name: ""amyotroph...",1,"MATCH (d:Disease {name: ""amyotrophic lateral s...",True,[],0.142036,0.0,NaN
3,mistral_base,"(tdp-als, What (or how strong, or is there any...","\n\n```cypher\nMATCH (d:Disease {name: ""amyotr...","[{'query': 'MATCH (d:Disease {name: ""amyotroph...",1,"MATCH (d:Disease {name: ""amyotrophic lateral s...",True,[],0.141532,0.0,NaN
4,mistral_base,"(tdp-als, What (or how strong, or is there any...","\n\n```cypher\nMATCH (d:Disease {name: ""amyotr...","[{'query': 'MATCH (d:Disease {name: ""amyotroph...",1,"MATCH (d:Disease {name: ""amyotrophic lateral s...",True,[],0.140060,0.0,NaN
5,mistral_base,"(tdp-als, What (or how strong, or is there any...","\n\n```cypher\nMATCH (d:Disease {name: ""amyotr...","[{'query': 'MATCH (d:Disease {name: ""amyotroph...",1,"MATCH (d:Disease {name: ""amyotrophic lateral s...",True,[],0.145514,0.0,NaN
6,mistral_base,"(tdp-als, What (or how strong, or is there any...","\n\n```cypher\nMATCH (d:Disease {name: ""amyotr...","[{'query': 'MATCH (d:Disease {name: ""amyotroph...",1,"MATCH (d:Disease {name: ""amyotrophic lateral s...",True,[],0.143522,0.0,NaN
7,mistral_base,"(tdp-als, What (or how strong, or is there any...","\n\n```cypher\nMATCH (d:Disease {name: ""amyotr...","[{'query': 'MATCH (d:Disease {name: ""amyotroph...",1,"MATCH (d:Disease {name: ""amyotrophic lateral s...",True,[],0.145642,0.0,NaN
8,mistral_base,"(tdp-als, What (or how strong, or is there any...","\n\n```cypher\nMATCH (d:Disease {name: ""amyotr...","[{'query': 'MATCH (d:Disease {name: ""amyotroph...",1,"MATCH (d:Disease {name: ""amyotrophic lateral s...",True,[],0.139516,0.0,NaN
9,mistral_base,"(tdp-als, What (or how strong, or is there any...","\n\n```cypher\nMATCH (d:Disease {name: ""amyotr...","[{'query': 'MATCH (d:Disease {name: ""amyotroph...",1,"MATCH (d:Disease {name: ""amyotrophic lateral s...",True,[],0.143040,0.0,NaN
